# Triage Pilot — Phase 0 Checkpoint 4: Gemma 4 E2B Baseline

**Purpose:** Zero-shot Gemma 4 E2B across the frozen test set for all four SAM tasks. This is the deployable generalist baseline — the thing SAMs need to beat on latency and cost while staying close on accuracy.

**Runs on:** Colab T4 (E2B fits comfortably in FP16 on 16GB). Vertex L4 also fine.

**Output:** writes results into `baselines/baseline_results.json` under the key `gemma_e2b`.

**Note:** this notebook is structurally identical to `phase0_baselines_e4b.ipynb` — only `MODEL_ID` and `MODEL_KEY` differ. Keep them in sync if you change the runner logic.

## 1. Config

In [ ]:
import os

TRIAGE_ROOT = '/content/drive/MyDrive/setique/triage'
DATA_PROCESSED_DIR = f'{TRIAGE_ROOT}/data/processed'
EVAL_DIR = f'{TRIAGE_ROOT}/eval'
BASELINES_DIR = f'{TRIAGE_ROOT}/baselines'
PROMPTS_DIR = f'{BASELINES_DIR}/prompts'

TEST_PATH = f'{DATA_PROCESSED_DIR}/test.jsonl'
HASH_PATH = f'{EVAL_DIR}/test_set_hash.txt'
RESULTS_PATH = f'{BASELINES_DIR}/baseline_results.json'

MODEL_KEY = 'gemma_e2b'
# Update the HF model ID once the Gemma 4 E2B release name is confirmed on the hub
MODEL_ID = 'google/gemma-4-2b-it'  # ← placeholder; verify before running

LIMIT = None

## 2. Mount + imports

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    print('Not running in Colab — assuming TRIAGE_ROOT is already accessible.')

!pip install -q transformers accelerate bitsandbytes scikit-learn

In [ ]:
import sys
sys.path.insert(0, f'{TRIAGE_ROOT}/../')

from baselines_runner import (
    verify_test_hash, load_test_set, load_prompt,
    run_inference, merge_results,
    peak_vram_gb, reset_vram_tracking,
)

## 3. Verify frozen test set

In [ ]:
test_hash = verify_test_hash(TEST_PATH, HASH_PATH)
records = load_test_set(TEST_PATH)
print(f'Test hash verified: {test_hash}')
print(f'Loaded {len(records)} test records')

## 4. Load Gemma 4 E2B

On T4 use FP16 (no native BF16). On L4/A100 BF16 is fine.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

reset_vram_tracking()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
preferred_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=preferred_dtype,
    device_map='auto',
)
model.eval()
print(f'Loaded {MODEL_ID} in {preferred_dtype}. Peak VRAM after load: {peak_vram_gb():.2f} GB')

## 5. Generate function

In [ ]:
@torch.inference_mode()
def generate(prompt: str, max_new_tokens: int = 16) -> str:
    messages = [{'role': 'user', 'content': prompt}]
    try:
        input_ids = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors='pt'
        ).to(model.device)
    except Exception:
        input_ids = tokenizer(prompt, return_tensors='pt').input_ids.to(model.device)

    out = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=1.0,
        pad_token_id=tokenizer.eos_token_id,
    )
    new_tokens = out[0][input_ids.shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

print('Smoke test output:', generate('Respond with the single word: ready'))

## 6. Run all four SAM tasks

In [ ]:
LANG_CODES = {'en','es','fr','de','pt','it','nl','ja','zh','ko',
              'ar','ru','pl','tr','sv','hi','id','vi','th','other'}
INTENT_LABELS = {
    'billing_question','billing_dispute','refund_request','cancel_subscription',
    'password_reset','account_access','update_profile','delete_account',
    'how_to','feature_request','bug_report','compatibility_question',
    'order_status','shipping_question','return_request','damaged_item',
    'technical_issue','escalation_request','complaint','compliment',
    'spam','unclear','other',
}
PRIORITY_LABELS = {'P1','P2','P3','P4'}
ROUTE_LABELS = {
    'billing_team','technical_support','account_security',
    'returns_and_shipping','product_feedback','legal_compliance',
    'human_escalation','auto_close',
}

TASKS = [
    ('sam_lang',     'lang',     LANG_CODES,     'other'),
    ('sam_intent',   'intent',   INTENT_LABELS,  'other'),
    ('sam_priority', 'priority', PRIORITY_LABELS,'P3'),
    ('sam_route',    'route',    ROUTE_LABELS,   'human_escalation'),
]

results_per_sam = {}
for sam_name, target_field, valid_labels, fallback in TASKS:
    print(f'\n=== {sam_name} ===')
    prompt_path = f'{PROMPTS_DIR}/{sam_name}.txt'
    prompt_template = load_prompt(prompt_path)
    reset_vram_tracking()
    result = run_inference(
        records=records,
        prompt_template=prompt_template,
        generate_fn=generate,
        valid_labels=valid_labels,
        target_field=target_field,
        fallback=fallback,
        limit=LIMIT,
        verbose=True,
    )
    result['peak_vram_gb'] = peak_vram_gb()
    results_per_sam[sam_name] = result
    print(f'  accuracy:   {result["accuracy"]:.4f}')
    print(f'  f1_macro:   {result["f1_macro"]:.4f}')
    print(f'  latency p50:{result["latency_ms_p50"]:.0f} ms')
    print(f'  latency p95:{result["latency_ms_p95"]:.0f} ms')
    print(f'  fallback %: {result["parse_fallback_rate"]*100:.2f}%')
    print(f'  peak VRAM:  {result["peak_vram_gb"]:.2f} GB')

## 7. Persist results

In [ ]:
from datetime import datetime

metadata = {
    'model_id': MODEL_ID,
    'dtype': str(preferred_dtype).split('.')[-1],
    'test_hash': test_hash,
    'n_test': len(records),
    'limit': LIMIT,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'timestamp_utc': datetime.utcnow().isoformat() + 'Z',
    'decode': 'greedy, max_new_tokens=16',
}

merge_results(RESULTS_PATH, MODEL_KEY, results_per_sam, metadata)
print(f'Results merged into {RESULTS_PATH} under key "{MODEL_KEY}"')

## 8. Cost estimate

In [ ]:
# Colab T4 (Pro) ~$0.09/hr compute-equivalent is a very rough pro-rata
# L4 on Vertex ~$0.75/hr. Update to match actual instance.
INSTANCE_HOURLY_USD = 0.75

for sam_name, res in results_per_sam.items():
    mean_sec = res['latency_ms_mean'] / 1000.0
    cost_per_1k = mean_sec * 1000 / 3600 * INSTANCE_HOURLY_USD
    print(f'{sam_name:14s}  ${cost_per_1k:.4f} per 1K inferences')

## 9. Checkpoint 4 verdict

- [ ] Both `gemma_e4b` and `gemma_e2b` keys populated in `baseline_results.json`
- [ ] Parse fallback rates reasonable on both
- [ ] E4B > E2B on accuracy (sanity check — if not, something's wrong with the larger model's prompt or setup)
- [ ] Latency and VRAM numbers recorded for the TCO story
- [ ] Commit `baseline_results.json` to the repo

**Once all four checkpoints are green, Phase 0 is done.** Move to Phase 1 (SAM-Lang training) — it's the lowest-risk of the four SAMs and gives you a clean win to validate the NICO plumbing before tackling SAM-Intent.